# Esercitazione PySpark: analisi delle vendite 2019-2021

Questo notebook introduce **PySpark da zero** attraverso un caso pratico: analizzare gli ordini di vendita del periodo 2019-2021.

Il percorso è progettato per un workshop di circa **2-3 ore** in Google Colab.

## Obiettivi

Al termine dell'esercitazione saprai:

- creare una `SparkSession`;
- leggere file CSV applicando uno schema esplicito;
- esplorare, selezionare, filtrare e ordinare i dati;
- gestire valori nulli e creare colonne calcolate;
- unire DataFrame e produrre aggregazioni;
- creare temporary view e interrogarle con Spark SQL;
- usare il catalogo Spark per individuare le view disponibili;
- confrontare DataFrame API e Spark SQL.


## 1. Preparazione dell'ambiente

### Installazione di PySpark

Google Colab include Java, ma PySpark deve essere installato. Fissiamo la versione per rendere l'esercitazione riproducibile. Dopo l'installazione Colab potrebbe richiedere qualche secondo prima dell'importazione.


In [ ]:
%pip install -q pyspark==3.5.3


### Importazione delle librerie

Importiamo i tipi necessari per definire lo schema e le funzioni PySpark che useremo. L'alias `F` rende riconoscibili le funzioni Spark nel codice.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    DecimalType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)


### Creazione della SparkSession

La `SparkSession` è il punto di ingresso principale per lavorare con DataFrame e Spark SQL. `getOrCreate()` riutilizza una sessione esistente oppure ne crea una nuova.


In [ ]:
spark = (
    SparkSession.builder
    .appName("Esercitazione PySpark - Vendite")
    .getOrCreate()
)

print(f"Versione Spark: {spark.version}")


### Concetti essenziali: trasformazioni e azioni

Spark applica la **lazy evaluation**:

- le trasformazioni, come `select()` e `filter()`, definiscono un piano di lavoro;
- le azioni, come `show()` e `count()`, avviano realmente il calcolo.

Questa distinzione diventerà visibile negli esempi successivi.


## 2. Caricamento dei file

### Upload dei dataset in Colab

Esegui la cella e seleziona insieme `2019.csv`, `2020.csv` e `2021.csv`. Se i file sono già presenti nella sessione Colab, non verrà richiesto un nuovo upload.


In [ ]:
from google.colab import files

required_files = {"2019.csv", "2020.csv", "2021.csv"}
missing_files = sorted(name for name in required_files if not Path(name).exists())

if missing_files:
    print("Seleziona i file:", ", ".join(missing_files))
    files.upload()
else:
    print("I file richiesti sono già disponibili.")


### Verifica dei file

Prima di proseguire controlliamo che tutti i dataset siano disponibili. L'`assert` interrompe subito l'esecuzione e mostra chiaramente gli eventuali file mancanti.


In [ ]:
missing_files = sorted(name for name in required_files if not Path(name).exists())
assert not missing_files, f"File mancanti: {missing_files}"

for name in sorted(required_files):
    print(f"{name}: {Path(name).stat().st_size:,} byte")


## 3. Schema e lettura dei CSV

### Definizione dello schema

I file degli ordini **non contengono una riga di intestazione**. Definiamo quindi nomi e tipi delle colonne manualmente.

Per prezzi e imposte utilizziamo `DecimalType` invece di `FloatType`, perché i numeri decimali sono più adatti ai valori monetari.


In [ ]:
order_schema = StructType([
    StructField("SalesOrderNumber", StringType(), nullable=False),
    StructField("SalesOrderLineNumber", IntegerType(), nullable=False),
    StructField("OrderDate", DateType(), nullable=False),
    StructField("CustomerName", StringType(), nullable=True),
    StructField("Email", StringType(), nullable=True),
    StructField("Item", StringType(), nullable=False),
    StructField("Quantity", IntegerType(), nullable=False),
    StructField("UnitPrice", DecimalType(12, 4), nullable=False),
    StructField("Tax", DecimalType(12, 4), nullable=False),
])

print(order_schema.simpleString())


### Funzione riutilizzabile per leggere un anno

La funzione applica sempre le stesse opzioni e aggiunge `OrderYear`, utile per confrontare gli anni. `header=False` è fondamentale: impostandolo a `True` perderemmo il primo ordine di ogni file.


In [ ]:
def read_orders(file_name: str, year: int):
    return (
        spark.read
        .option("header", False)
        .option("dateFormat", "yyyy-MM-dd")
        .schema(order_schema)
        .csv(file_name)
        .withColumn("OrderYear", F.lit(year))
    )


### Lettura dei tre dataset

Creiamo un DataFrame distinto per ogni anno. I nomi espliciti evitano di sovrascrivere accidentalmente una variabile generica come `df`.


In [ ]:
orders_2019 = read_orders("2019.csv", 2019)
orders_2020 = read_orders("2020.csv", 2020)
orders_2021 = read_orders("2021.csv", 2021)


### Controllo dello schema

`printSchema()` mostra la struttura interpretata da Spark. Verifica in particolare che `OrderDate` sia una data e che prezzi e imposte siano decimali.


In [ ]:
orders_2019.printSchema()


### Visualizzazione delle prime righe

`show()` è un'azione: Spark legge i dati necessari e visualizza un campione. `truncate=False` evita di abbreviare nomi e descrizioni.


In [ ]:
orders_2019.show(5, truncate=False)


### Conteggio degli ordini per anno

`count()` è un'altra azione. I conteggi permettono anche di verificare che la prima riga dei file non sia stata eliminata per errore.


In [ ]:
print("2019:", orders_2019.count())
print("2020:", orders_2020.count())
print("2021:", orders_2021.count())


### Esercizio 1: esplorare un DataFrame

Completa la cella per visualizzare dieci righe del 2020 senza troncare il testo e stampare il numero delle colonne.


In [ ]:
# ESERCIZIO
# orders_2020. ...
# print(...)


### Soluzione dell'esercizio 1

La proprietà `columns` restituisce una lista Python con i nomi delle colonne; `len()` ne calcola il numero.


In [ ]:
orders_2020.show(10, truncate=False)
print("Numero di colonne:", len(orders_2020.columns))


## 4. Unione e prima esplorazione

### Unione dei tre anni

`unionByName()` combina DataFrame con lo stesso schema allineando le colonne per nome. Il risultato rappresenta l'intero periodo 2019-2021.


In [ ]:
all_orders = (
    orders_2019
    .unionByName(orders_2020)
    .unionByName(orders_2021)
)

print("Righe complessive:", all_orders.count())


### Selezione delle colonne

`select()` crea un nuovo DataFrame senza modificare quello originale. Questa immutabilità consente di costruire trasformazioni progressive e controllabili.


In [ ]:
order_summary = all_orders.select(
    "SalesOrderNumber",
    "OrderDate",
    "CustomerName",
    "Item",
    "Quantity",
    "UnitPrice",
    "OrderYear",
)

order_summary.show(5, truncate=False)


### Filtraggio delle righe

`filter()` mantiene solamente gli ordini che soddisfano una condizione. Qui cerchiamo le righe con prezzo unitario almeno pari a 2.000.


In [ ]:
high_price_orders = all_orders.filter(F.col("UnitPrice") >= 2000)
high_price_orders.select("Item", "UnitPrice", "OrderYear").show(10, truncate=False)


### Ordinamento

`orderBy()` ordina le righe. Con `desc()` richiediamo i prezzi dal maggiore al minore.


In [ ]:
all_orders.select("Item", "UnitPrice", "OrderYear").orderBy(
    F.col("UnitPrice").desc()
).show(10, truncate=False)


### Condizioni multiple

Le condizioni Spark si combinano con `&` (AND), `|` (OR) e `~` (NOT). Ogni condizione deve essere racchiusa tra parentesi.


In [ ]:
orders_2021_expensive = all_orders.filter(
    (F.col("OrderYear") == 2021) &
    (F.col("UnitPrice") >= 1000)
)

orders_2021_expensive.show(5, truncate=False)


### Esercizio 2: selezione e filtro

Trova gli ordini del 2020 con quantità maggiore di uno. Mostra soltanto numero ordine, prodotto, quantità e prezzo unitario.


In [ ]:
# ESERCIZIO
# result = all_orders.filter(...).select(...)
# result.show(truncate=False)


### Soluzione dell'esercizio 2

Prima filtriamo le righe, poi riduciamo le colonne con `select()`. Le trasformazioni possono essere concatenate.


In [ ]:
result = (
    all_orders
    .filter((F.col("OrderYear") == 2020) & (F.col("Quantity") > 1))
    .select("SalesOrderNumber", "Item", "Quantity", "UnitPrice")
)

result.show(truncate=False)


## 5. Qualità e pulizia dei dati

### Conteggio dei valori nulli

Per ogni colonna trasformiamo la condizione `isNull()` in 1 oppure 0 e sommiamo i risultati. Questa tecnica produce un rapido profilo di completezza.


In [ ]:
null_counts = all_orders.select([
    F.sum(F.when(F.col(column).isNull(), 1).otherwise(0)).alias(column)
    for column in all_orders.columns
])

null_counts.show(truncate=False)


### Ispezione dei clienti mancanti

Prima di correggere i dati è utile osservare le righe interessate. `isNull()` individua i valori realmente nulli.


In [ ]:
all_orders.filter(F.col("CustomerName").isNull()).show(10, truncate=False)


### Pulizia dei nomi cliente

Consideriamo mancanti sia i valori nulli sia le stringhe vuote o composte da soli spazi. Li sostituiamo con `Unknown`; per gli altri valori applichiamo `trim()`.


In [ ]:
clean_orders = all_orders.withColumn(
    "CustomerName",
    F.when(
        F.col("CustomerName").isNull() |
        (F.trim(F.col("CustomerName")) == ""),
        F.lit("Unknown"),
    ).otherwise(F.trim(F.col("CustomerName")))
)


### Verifica della pulizia

Un controllo dopo la trasformazione rende esplicito il risultato atteso: non devono rimanere clienti nulli o vuoti.


In [ ]:
remaining_missing_customers = clean_orders.filter(
    F.col("CustomerName").isNull() |
    (F.trim(F.col("CustomerName")) == "")
).count()

assert remaining_missing_customers == 0
print("Pulizia completata: nessun cliente nullo o vuoto.")


## 6. Colonne calcolate

### Calcolo degli importi

Creiamo tre misure:

- `NetAmount`: quantità moltiplicata per prezzo unitario;
- `TaxAmount`: imposta già presente nel file;
- `TotalAmount`: somma di imponibile e imposta.

Manteniamo valori decimali per evitare approssimazioni tipiche dei numeri floating point.


In [ ]:
enriched_orders = (
    clean_orders
    .withColumn("NetAmount", F.col("Quantity") * F.col("UnitPrice"))
    .withColumnRenamed("Tax", "TaxAmount")
    .withColumn("TotalAmount", F.col("NetAmount") + F.col("TaxAmount"))
)

enriched_orders.select(
    "Item", "Quantity", "UnitPrice", "NetAmount", "TaxAmount", "TotalAmount"
).show(5, truncate=False)


### Colonna condizionale

`when().otherwise()` è l'equivalente Spark di una struttura `if/else`. Classifichiamo gli ordini in base al totale della riga.


In [ ]:
enriched_orders = enriched_orders.withColumn(
    "OrderCategory",
    F.when(F.col("TotalAmount") >= 2000, "High Value")
    .when(F.col("TotalAmount") >= 500, "Medium Value")
    .otherwise("Standard"),
)

enriched_orders.groupBy("OrderCategory").count().show()


### Estrazione di informazioni dalla data

Le funzioni `year()` e `month()` lavorano su colonne di tipo data. Aggiungiamo il mese per le analisi temporali successive.


In [ ]:
enriched_orders = (
    enriched_orders
    .withColumn("OrderMonth", F.month("OrderDate"))
    .withColumn("OrderYearFromDate", F.year("OrderDate"))
)

enriched_orders.select("OrderDate", "OrderYear", "OrderYearFromDate", "OrderMonth").show(5)


### Controllo di coerenza dell'anno

Confrontiamo l'anno derivato dal nome del file con quello contenuto nella data. Il controllo deve restituire zero righe incoerenti.


In [ ]:
invalid_year_rows = enriched_orders.filter(
    F.col("OrderYear") != F.col("OrderYearFromDate")
).count()

assert invalid_year_rows == 0
print("Anno del file e anno della data sono coerenti.")


### Esercizio 3: creare una colonna

Aggiungi `UnitPriceBand` con i valori `Premium` per prezzi almeno pari a 1.000 e `Regular` negli altri casi.


In [ ]:
# ESERCIZIO
# exercise_orders = enriched_orders.withColumn(...)
# exercise_orders.groupBy("UnitPriceBand").count().show()


### Soluzione dell'esercizio 3

La nuova colonna viene aggiunta senza modificare `enriched_orders`; il conteggio permette di controllare la distribuzione delle categorie.


In [ ]:
exercise_orders = enriched_orders.withColumn(
    "UnitPriceBand",
    F.when(F.col("UnitPrice") >= 1000, "Premium").otherwise("Regular"),
)

exercise_orders.groupBy("UnitPriceBand").count().show()


## 7. Aggregazioni

### Fatturato per anno

`groupBy()` definisce i gruppi e `agg()` calcola una o più misure. Arrotondiamo il totale solamente per la presentazione.


In [ ]:
sales_by_year = (
    enriched_orders
    .groupBy("OrderYear")
    .agg(
        F.countDistinct("SalesOrderNumber").alias("Orders"),
        F.sum("Quantity").alias("Units"),
        F.round(F.sum("TotalAmount"), 2).alias("Revenue"),
    )
    .orderBy("OrderYear")
)

sales_by_year.show()


### Prodotti con maggiore fatturato

Raggruppiamo per prodotto, sommiamo quantità e ricavi e ordiniamo il risultato in modo decrescente.


In [ ]:
top_products = (
    enriched_orders
    .groupBy("Item")
    .agg(
        F.sum("Quantity").alias("Units"),
        F.round(F.sum("TotalAmount"), 2).alias("Revenue"),
    )
    .orderBy(F.col("Revenue").desc())
)

top_products.show(10, truncate=False)


### Clienti con maggiore spesa

Escludiamo `Unknown` dalla classifica per evitare che clienti distinti ma privi di nome vengano considerati come una sola persona.


In [ ]:
top_customers = (
    enriched_orders
    .filter(F.col("CustomerName") != "Unknown")
    .groupBy("CustomerName")
    .agg(
        F.countDistinct("SalesOrderNumber").alias("Orders"),
        F.round(F.sum("TotalAmount"), 2).alias("TotalSpent"),
    )
    .orderBy(F.col("TotalSpent").desc())
)

top_customers.show(10, truncate=False)


### Andamento mensile

Raggruppare per anno e mese evita di sommare insieme, per esempio, gennaio 2019 e gennaio 2020.


In [ ]:
monthly_sales = (
    enriched_orders
    .groupBy("OrderYear", "OrderMonth")
    .agg(F.round(F.sum("TotalAmount"), 2).alias("Revenue"))
    .orderBy("OrderYear", "OrderMonth")
)

monthly_sales.show(15)


### Esercizio 4: valore medio degli ordini

Calcola, per ogni anno, la media di `TotalAmount` e chiamala `AverageLineValue`. Ordina il risultato per anno.


In [ ]:
# ESERCIZIO
# average_value_by_year = enriched_orders.groupBy(...).agg(...).orderBy(...)
# average_value_by_year.show()


### Soluzione dell'esercizio 4

Questa media riguarda il valore delle righe d'ordine. Un ordine può contenere più righe: la distinzione è importante nell'interpretazione del risultato.


In [ ]:
average_value_by_year = (
    enriched_orders
    .groupBy("OrderYear")
    .agg(F.round(F.avg("TotalAmount"), 2).alias("AverageLineValue"))
    .orderBy("OrderYear")
)

average_value_by_year.show()


## 8. Temporary view e Spark SQL

### Creazione della view

`createOrReplaceTempView()` assegna un nome SQL al DataFrame. La view non duplica i dati: conserva il piano logico necessario a produrli.

Una temporary view è disponibile solamente nella `SparkSession` corrente.


In [ ]:
enriched_orders.createOrReplaceTempView("orders")
print("Temporary view 'orders' creata.")


### Esplorazione tramite il catalogo

Il catalogo Spark contiene i metadati di tabelle e view accessibili dalla sessione. `listTables()` permette di verificare che `orders` sia stata registrata.


In [ ]:
spark.catalog.listTables()


### Verifica dell'esistenza della view

`tableExists()` è utile prima di eseguire query che dipendono da una tabella o view.


In [ ]:
assert spark.catalog.tableExists("orders")
print("La view 'orders' è disponibile nel catalogo della sessione.")


### Prima query Spark SQL

`spark.sql()` esegue una query SQL e restituisce un nuovo DataFrame Spark. Possiamo quindi continuare a utilizzare metodi come `show()` sul risultato.


In [ ]:
sql_sample = spark.sql("""
    SELECT
        SalesOrderNumber,
        OrderDate,
        CustomerName,
        Item,
        TotalAmount
    FROM orders
    ORDER BY OrderDate
    LIMIT 10
""")

sql_sample.show(truncate=False)


### Aggregazione tramite Spark SQL

La query usa la stessa view per calcolare il fatturato per anno. Il backtick intorno a `Year` non è necessario perché utilizziamo il nome `OrderYear`, non riservato.


In [ ]:
sql_sales_by_year = spark.sql("""
    SELECT
        OrderYear,
        COUNT(DISTINCT SalesOrderNumber) AS Orders,
        SUM(Quantity) AS Units,
        ROUND(SUM(TotalAmount), 2) AS Revenue
    FROM orders
    GROUP BY OrderYear
    ORDER BY OrderYear
""")

sql_sales_by_year.show()


### Confronto tra DataFrame API e SQL

Entrambe le API vengono tradotte in piani Spark. `explain()` permette di osservare il piano fisico scelto dal motore.


In [ ]:
print("PIANO DATAFRAME API")
sales_by_year.explain()

print("\nPIANO SPARK SQL")
sql_sales_by_year.explain()


### Query con CTE

Una Common Table Expression (`WITH`) rende più leggibili le query articolate. Calcoliamo prima il fatturato per cliente e poi selezioniamo i primi dieci.


In [ ]:
sql_top_customers = spark.sql("""
    WITH customer_sales AS (
        SELECT
            CustomerName,
            COUNT(DISTINCT SalesOrderNumber) AS Orders,
            ROUND(SUM(TotalAmount), 2) AS TotalSpent
        FROM orders
        WHERE CustomerName <> 'Unknown'
        GROUP BY CustomerName
    )
    SELECT CustomerName, Orders, TotalSpent
    FROM customer_sales
    ORDER BY TotalSpent DESC
    LIMIT 10
""")

sql_top_customers.show(truncate=False)


### Creazione di una seconda view

Anche il risultato di una query può diventare una view. Registriamo le vendite mensili per riutilizzarle senza riscrivere l'aggregazione.


In [ ]:
monthly_sales.createOrReplaceTempView("monthly_sales")

spark.sql("""
    SELECT *
    FROM monthly_sales
    WHERE OrderYear = 2021
    ORDER BY OrderMonth
""").show()


### Esercizio 5: interrogare una view

Scrivi una query SQL sulla view `orders` che restituisca i cinque prodotti con più unità vendute. Usa `SUM(Quantity)`, `GROUP BY`, `ORDER BY` e `LIMIT`.


In [ ]:
# ESERCIZIO
# spark.sql("""
#     SELECT ...
#     FROM orders
#     ...
# """).show(truncate=False)


### Soluzione dell'esercizio 5

L'alias `Units` può essere usato direttamente nella clausola `ORDER BY`.


In [ ]:
spark.sql("""
    SELECT
        Item,
        SUM(Quantity) AS Units
    FROM orders
    GROUP BY Item
    ORDER BY Units DESC
    LIMIT 5
""").show(truncate=False)


### Rimozione di una view

`dropTempView()` elimina il riferimento dal catalogo, non i dati originali. Manteniamo `orders` per il mini-progetto e rimuoviamo la view intermedia.


In [ ]:
removed = spark.catalog.dropTempView("monthly_sales")
print("View rimossa:", removed)
print("monthly_sales esiste ancora?", spark.catalog.tableExists("monthly_sales"))


## 9. Mini-progetto conclusivo

Utilizza la view `orders` per creare un report annuale con:

- anno;
- numero di ordini distinti;
- unità vendute;
- fatturato totale;
- valore medio delle righe;
- percentuale di fatturato rispetto all'intero periodo.

Ordina il risultato cronologicamente. Prova a costruire autonomamente la query prima di aprire la soluzione.


### Area di lavoro del mini-progetto

Scrivi la query all'interno della stringa passata a `spark.sql()`.


In [ ]:
# MINI-PROGETTO
# final_report = spark.sql("""
#     WITH ...
#     SELECT ...
# """)
# final_report.show()


### Soluzione del mini-progetto

La prima CTE calcola le misure annuali; la seconda calcola il fatturato complessivo. Il `CROSS JOIN` rende disponibile il totale su ogni riga per il calcolo percentuale.


In [ ]:
final_report = spark.sql("""
    WITH yearly AS (
        SELECT
            OrderYear,
            COUNT(DISTINCT SalesOrderNumber) AS Orders,
            SUM(Quantity) AS Units,
            SUM(TotalAmount) AS Revenue,
            AVG(TotalAmount) AS AverageLineValue
        FROM orders
        GROUP BY OrderYear
    ),
    overall AS (
        SELECT SUM(Revenue) AS OverallRevenue
        FROM yearly
    )
    SELECT
        y.OrderYear,
        y.Orders,
        y.Units,
        ROUND(y.Revenue, 2) AS Revenue,
        ROUND(y.AverageLineValue, 2) AS AverageLineValue,
        ROUND(y.Revenue / o.OverallRevenue * 100, 2) AS RevenuePercentage
    FROM yearly AS y
    CROSS JOIN overall AS o
    ORDER BY y.OrderYear
""")

final_report.show()


### Controlli finali

Verifichiamo automaticamente che il report contenga i tre anni previsti e che le percentuali di fatturato sommino circa a 100.


In [ ]:
report_rows = final_report.collect()
report_years = {row["OrderYear"] for row in report_rows}
percentage_total = sum(float(row["RevenuePercentage"]) for row in report_rows)

assert report_years == {2019, 2020, 2021}
assert abs(percentage_total - 100.0) <= 0.1
print("Report finale validato.")


## 10. Conclusioni

In questa esercitazione abbiamo costruito un flusso completo:

1. definizione dello schema;
2. lettura e unione dei CSV;
3. esplorazione e pulizia;
4. creazione di misure;
5. aggregazioni con DataFrame API;
6. registrazione e interrogazione di temporary view;
7. consultazione del catalogo Spark;
8. produzione di un report SQL.

### Approfondimenti suggeriti

- window functions;
- join tra dataset;
- lettura e scrittura in formato Parquet o Delta;
- partizionamento e caching;
- cataloghi persistenti in Databricks o Microsoft Fabric.


### Chiusura della sessione

Al termine del lavoro possiamo rimuovere la view e arrestare la sessione. Esegui questa cella soltanto quando non devi più utilizzare i DataFrame del notebook.


In [ ]:
spark.catalog.dropTempView("orders")
spark.stop()
print("SparkSession arrestata.")
